In [2]:
import pandas as pd
import numpy as np
import re

# 1) Read datasets
cosmetics = pd.read_csv("cosmetics.csv")
product_info = pd.read_csv("product_info.csv")

# 2) Cleaning function
def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = text.strip()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text

# 3) Create matching keys
cosmetics["brand_clean"] = cosmetics["Brand"].apply(clean_text)
cosmetics["product_clean"] = cosmetics["Name"].apply(clean_text)

product_info["brand_clean"] = product_info["brand_name"].apply(clean_text)
product_info["product_clean"] = product_info["product_name"].apply(clean_text)

# 4) Select needed columns
cosmetics_selected = cosmetics[
    [
        "brand_clean",
        "product_clean",
        "Brand",
        "Name",
        "Label",
        "Rank",
        "Dry",
        "Normal",
        "Oily",
        "Sensitive"
    ]
].copy()

product_selected = product_info[
    [
        "brand_clean",
        "product_clean",
        "brand_name",
        "product_name",
        "loves_count",
        "rating",
        "price_usd"
    ]
].copy()

# 5) Convert numeric columns
cosmetics_selected["Rank"] = pd.to_numeric(cosmetics_selected["Rank"], errors="coerce")
product_selected["rating"] = pd.to_numeric(product_selected["rating"], errors="coerce")
product_selected["price_usd"] = pd.to_numeric(product_selected["price_usd"], errors="coerce")
product_selected["loves_count"] = pd.to_numeric(product_selected["loves_count"], errors="coerce")

# 6) Merge datasets based on same brand + product name
merged = pd.merge(
    cosmetics_selected,
    product_selected,
    on=["brand_clean", "product_clean"],
    how="inner",
    suffixes=("_cosmetics", "_product")
)

print("Number of matched products:", merged.shape[0])

# 7) Average rating from both datasets
merged["avg_rating"] = merged[["Rank", "rating"]].mean(axis=1)

# 8) Create final dataset
final_df = merged[
    [
        "brand_name",
        "product_name",
        "loves_count",
        "avg_rating",
        "Label",
        "price_usd",
        "Dry",
        "Normal",
        "Oily",
        "Sensitive"
    ]
].copy()

# 9) Rename columns
final_df = final_df.rename(columns={
    "brand_name": "brand",
    "product_name": "product_name",
    "loves_count": "loves_count",
    "avg_rating": "rating",
    "Label": "label",
    "price_usd": "price",
    "Dry": "dry",
    "Normal": "normal",
    "Oily": "oily",
    "Sensitive": "sensitive"
})

# 10) Remove missing important values
final_df = final_df.dropna(subset=["price", "rating", "loves_count"])

# 11) Remove duplicates
final_df = final_df.drop_duplicates(subset=["brand", "product_name"], keep="first")

# 12) Add log features
final_df["log_price"] = np.log1p(final_df["price"])
final_df["log_loves_count"] = np.log1p(final_df["loves_count"])

# 13) Check final result
print("Final dataset shape:", final_df.shape)
print(final_df.isnull().sum())

display(final_df.head())

# 14) Save dataset
final_df.to_csv("merged_cosmetic_product_dataset.csv", index=False)

print("Saved as merged_cosmetic_product_dataset.csv")

Number of matched products: 295
Final dataset shape: (293, 12)
brand              0
product_name       0
loves_count        0
rating             0
label              0
price              0
dry                0
normal             0
oily               0
sensitive          0
log_price          0
log_loves_count    0
dtype: int64


,brand,product_name,loves_count,rating,label,price,dry,normal,oily,sensitive,log_price,log_loves_count
0,belif,The True Cream Aqua Bomb,265050,4.49205,Moisturizer,38.0,0,1,1,0,3.663562,12.487678
1,First Aid Beauty,Ultra Repair Cream Intense Hydration,300432,4.56000,Moisturizer,38.0,1,1,1,1,3.663562,12.612980
2,Shiseido,Bio-Performance Advanced Super Revitalizing Cream,22311,4.56675,Moisturizer,85.0,0,0,0,0,4.454347,10.012880
3,fresh,Black Tea Firming Overnight Mask,73168,4.11845,Moisturizer,96.0,1,1,0,0,4.574711,11.200527
4,belif,The True Cream Moisturizing Bomb,151868,4.58155,Moisturizer,38.0,1,1,0,0,3.663562,11.930774


Saved as merged_cosmetic_product_dataset.csv


In [3]:
"""
Ethical Elephant - Sephora Cruelty-Free Brands Scraper
Setup:  pip install requests beautifulsoup4 pandas openpyxl
Run:    python scrape_cruelty_free_final.py

Output columns:
  Brand       - brand name
  Brand_clean - normalized name for merging
  CF          - 1 if cruelty-free (including parent company cases), 0 if not
"""

import re
import requests
from bs4 import BeautifulSoup
import pandas as pd

URL = "https://ethicalelephant.com/sephora-cruelty-free-brands-list/"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}


def get_cf(img) -> int:
    """
    Returns 1 if cruelty-free (or cruelty-free but parent company).
    Returns 0 if tests on animals or grey area.
    """
    src = (img.get("data-src") or img.get("src") or "").lower()
    if "tests-on-animals" in src:
        return 0
    if "grey-area" in src or "uncertain" in src:
        return 0
    if "cruelty-free" in src:   # covers both cruelty-free and cruelty-free-but
        return 1
    return 0


def get_brand_name(li) -> str:
    # Remove noscript and span tags before extracting name
    for tag in li.find_all(["noscript", "span"]):
        tag.decompose()

    # <del> tag = crossed out = tests on animals
    del_tag = li.find("del")
    if del_tag:
        return del_tag.get_text(strip=True)

    # <a> tag = normal brand link
    a_tag = li.find("a")
    if a_tag:
        return a_tag.get_text(strip=True)

    return li.get_text(separator=" ", strip=True)


def scrape() -> pd.DataFrame:
    print(f"Fetching: {URL}")
    resp = requests.get(URL, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    brands = []

    for li in soup.find_all("li"):
        img = li.find("img", class_=re.compile(r"wp-image"))
        if not img:
            continue

        cf = get_cf(img)
        brand_name = get_brand_name(li)
        brand_name = re.sub(r"\s+", " ", brand_name).strip()

        if not brand_name or len(brand_name) < 2:
            continue

        brand_clean = re.sub(r"[^A-Z0-9 ]", " ", brand_name.upper())
        brand_clean = re.sub(r"\s+", " ", brand_clean).strip()

        brands.append({
            "Brand":       brand_name,
            "Brand_clean": brand_clean,
            "CF":          cf,
        })

    df = pd.DataFrame(brands).drop_duplicates(subset="Brand_clean").reset_index(drop=True)
    return df


def main():
    df = scrape()

    if df.empty:
        print("No brands found!")
        return

    print(f"\nTotal brands: {len(df)}")
    print(f"  Cruelty-free (CF=1): {(df['CF'] == 1).sum()}")
    print(f"  Not cruelty-free (CF=0): {(df['CF'] == 0).sum()}\n")
    print(df.to_string(index=False))

    df.to_excel("cruelty_free_brands.xlsx", index=False)
    df.to_csv("cruelty_free_brands.csv", index=False, encoding="utf-8-sig")
    print("\nSaved: cruelty_free_brands.xlsx and .csv")


if __name__ == "__main__":
    main()

Fetching: https://ethicalelephant.com/sephora-cruelty-free-brands-list/

Total brands: 260
  Cruelty-free (CF=1): 189
  Not cruelty-free (CF=0): 71

                     Brand                Brand_clean  CF
                    Abbott                     ABBOTT   1
                  Act+Acre                   ACT ACRE   1
              adwoa beauty               ADWOA BEAUTY   1
                     AERIN                      AERIN   0
                  Algenist                   ALGENIST   0
                       ALO                        ALO   1
              Alpyn Beauty               ALPYN BEAUTY   1
          Alterna Haircare           ALTERNA HAIRCARE   0
                 Ami ColÃ©                    AMI COL   1
                     amika                      AMIKA   1
   Anastasia Beverly Hills    ANASTASIA BEVERLY HILLS   1
                     Aquis                      AQUIS   1
             Armani Beauty              ARMANI BEAUTY   0
            Artist Couture             

In [6]:
import pandas as pd
import re

# 1) Read existing merged dataset and cruelty-free data
final_df = pd.read_csv("merged_cosmetic_product_dataset.csv")
CF = pd.read_csv("cruelty_free_brands.csv")

# 2) Normalize brand names for matching
def clean_brand(text):
    if pd.isna(text):
        return ""
    text = str(text).upper()
    text = re.sub(r"[^A-Z0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

final_df["brand_clean"] = final_df["brand"].apply(clean_brand)
CF["brand_clean"] = CF["Brand_clean"].apply(clean_brand)

# 3) Merge CF column
cf_lookup = CF[["brand_clean", "CF"]].drop_duplicates(subset="brand_clean")

final_df = pd.merge(
    final_df,
    cf_lookup,
    on="brand_clean",
    how="left"
)

# 4) Brands not in the list → 0
final_df["CF"] = final_df["CF"].fillna(0).astype(int)

# 5) Drop helper column
final_df = final_df.drop(columns=["brand_clean"])

# 6) Check
print("CF value counts:")
print(final_df["CF"].value_counts())
print("\nUnmatched brands (CF=0):")
print(final_df[final_df["CF"] == 0]["brand"].unique())

# 7) Save
final_df.to_csv("merged_cosmetic_product_dataset.csv", index=False)
print("\nSaved: merged_cosmetic_product_dataset.csv")

CF value counts:
CF
0    194
1     99
Name: count, dtype: int64

Unmatched brands (CF=0):
['belif' 'Shiseido' 'fresh' 'Origins' 'CLINIQUE' 'SK-II' 'La Mer'
 'Bobbi Brown' 'Dr. Jart+' 'Algenist' "Kiehl's Since 1851"
 'REN Clean Skincare' "L'Occitane" 'Dr. Brandt Skincare' 'Dior' 'Lancôme'
 'SEPHORA COLLECTION' 'Peter Thomas Roth' 'Caudalie' 'MAKE UP FOR EVER'
 'Clarins' 'Estée Lauder' 'Fenty Beauty by Rihanna' 'Skin Laundry'
 'Naturally Serious' 'Peace Out' 'FOREO' 'NARS']

Saved: merged_cosmetic_product_dataset.csv
